# 05. Vasúti Diszkont és Izokrón Elemzés

**Cél**: A vasúti nyomvonal (zaj, rezgés, légszennyezés) negatív környezeti externáliájának, valamint a gyalogos izokrón zónák (TOD csomópontok) elérhetőségi prémiumának empirikus szétválasztása és kvantitatív vizsgálata Kőbányán.

**Módszertani alapelvek és nemzetközi standardok**:
1. **Környezeti terhek (zaj, rezgés, por)**: **Légvonalbeli (euklideszi)** távolság a vágánytengelytől (`tavolsag_vasut_m`), mivel a fizikai hullámok a térben és talajban radiálisan terjednek.
   - *Nemzetközi immissziós sávok (EU Noise Directive 2002/49/EC / WHO)*: `<150 m` (Közvetlen immisszió), `150–300 m` (Erős terhelés), `300–500 m` (Átmeneti sáv), `500–1000 m` (Városi háttérzaj), `1000–2000 m` (Közepes referencia), `>2000 m` (Tiszta referencia).
2. **Közlekedési elérhetőség (TOD)**: **Hálózati** távolság az OpenStreetMap gyalogos úthálózatán (Dijkstra algoritmus, $v = 1.25\text{ m/s} = 4.5\text{ km/h}$).
   - *Nemzetközi gyalogos izokrón sávok*: $\le 5$ perc ($\le 375\text{ m}$), $5–10$ perc ($375–750\text{ m}$), $10–15$ perc ($750–1125\text{ m}$), $>15$ perc ($>1125\text{ m}$).
3. **Horgonypontok funkcionális szétválasztása**:
   - *Kőbánya alsó vasútállomás*: Meglévő közlekedési csomópont (ingázás, TOD elérhetőségi prémium).
   - *Mázsa tér*: Városfejlesztési akcióterület (fejlesztési externália, LVC megtérülés).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from scipy import stats

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett eladó lakások száma: {len(elado)} db.")

### 1. Vasúti Puffer Zónák és Árdiszkont KPI-k
A vasúti pályatesttől mért légvonalbeli távolság alapján képzett nemzetközi standard környezeti immissziós sávok összehasonlítása.

In [ ]:
# Nemzetközi standard környezeti sávok (légvonalbeli távolság a vágányoktól)
elado['vasut_zona'] = pd.cut(
    elado['tavolsag_vasut_m'],
    bins=[0, 150, 300, 500, 1000, 2000, 10000],
    labels=['<150 m (Immisszió)', '150-300 m (Erős teher)', '300-500 m (Átmeneti)', '500-1000 m (Háttérzaj)', '1000-2000 m (Közepes ref.)', '>2000 m (Tiszta ref.)']
)

med_under150 = elado[elado['vasut_zona'] == '<150 m (Immisszió)']['nm_ar_huf'].median()
med_ref = elado[elado['vasut_zona'].isin(['1000-2000 m (Közepes ref.)', '>2000 m (Tiszta ref.)'])]['nm_ar_huf'].median()
vasut_diszkont_pct = ((med_under150 - med_ref) / med_ref) * 100 if pd.notna(med_ref) and med_ref > 0 else 0
n_terhelt_300 = int((elado['tavolsag_vasut_m'] < 300).sum())

kpi_cards = [
    ("<150m Immisszió Ár", fmt_huf(med_under150), "Közvetlen vasút menti", "#ef4444"),
    ("Referencia Zóna (>1km)", fmt_huf(med_ref), "Csendes övezeti ár", "#10b981"),
    ("Vasúti Árdiszkont", f"{vasut_diszkont_pct:.1f}%", "Immissziós árkülönbözet", "#dc2626"),
    ("Közvetlen Terhelt (<300m)", f"{n_terhelt_300} db", "Zaj- és rezgészóna", "#f59e0b"),
    ("Átlagos Légvonal Táv", f"{elado['tavolsag_vasut_m'].mean():.0f} m", "Vágányoktól mért táv", "#2563eb"),
    ("Állomás 10p Séta (750m)", f"{(elado['vasut_10p_seta'] == 1).sum()} db", "Kőbánya alsó vonzáskörzet", "#8b5cf6")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Vasúti Környezeti Távolság-Ár Gradiens (Légvonalban)
A négyzetméterár alakulása a vasúttól mért légvonalbeli fizikai távolság függvényében és az akusztikai lecsengési küszöb.

In [ ]:
fig1 = px.box(
    elado.dropna(subset=['vasut_zona']),
    x='vasut_zona',
    y='nm_ar_huf',
    color='vasut_zona',
    title='Négyzetméterár a nemzetközi standard vasúti környezeti immissziós sávok szerint (légvonalban)',
    labels={'vasut_zona': 'Vasúti környezeti sáv (légvonal)', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-25, height=450, showlegend=False)
fig1.show()

# Nem-parametrikus Kruskal-Wallis rangösszeg próba a 6 zóna közötti árkülönbségre
kw_groups = [g['nm_ar_huf'].values for _, g in elado.dropna(subset=['vasut_zona']).groupby('vasut_zona', observed=True)]
kw_stat, kw_p = stats.kruskal(*kw_groups)
display(HTML(f"<div style='background:#f1f5f9; padding:12px 18px; border-radius:8px; border-left:4px solid #2563eb; margin:12px 0;'>"
             f"<b>Kruskal–Wallis rangösszeg próba (6 immissziós zóna):</b> H = <b>{kw_stat:.2f}</b>, p-érték = <b>{kw_p:.4e}</b> "
             f"(Statisztikailag szignifikáns különbség a nemzetközi környezeti immissziós sávok fajlagos árai között).</div>"))

# Nem-lineáris távolsági gradiens scatter diagram légvonalbeli távolsággal
fig2 = px.scatter(
    elado,
    x='tavolsag_vasut_m',
    y='nm_ar_huf',
    color='varosresz',
    trendline='lowess',
    title='Légvonalbeli vasúttávolság vs. Négyzetméterár (LOWESS akusztikai lecsengési görbével)',
    labels={'tavolsag_vasut_m': 'Légvonalbeli távolság a vágányoktól (méter)', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=480)
fig2.show()

### 3. Gyalogos Izokrón Zónák és Elérhetőségi Prémiumok (Hálózaton)
A Kőbánya alsó vasútállomás, a Mázsa tér és a metróállomások 5p (≤375m), 10p (≤750m) és 15p (≤1125m) hálózati gyalogos izokrón zónáinak összehasonlítása.

In [ ]:
# Izokrón statisztikák összegzése a 375m (5p), 750m (10p), 1125m (15p) hálózati sávokra
izokron_adatok = []
celpontok = [
    ('Kőbánya alsó vasútállomás (TOD)', 'vasut'),
    ('Mázsa tér akcióterület (LVC)', 'mazsa'),
    ('Legközelebbi metróállomás', 'metro')
]

for cel_nev, col_prefix in celpontok:
    for p, m_dist in [(5, '≤375 m (5 perc)'), (10, '≤750 m (10 perc)'), (15, '≤1125 m (15 perc)')]:
        col = f'{col_prefix}_{p}p_seta'
        if col in elado.columns:
            minta = elado[elado[col] == 1]
            izokron_adatok.append({
                'Csomópont': cel_nev,
                'Gyalogos Izokrón': m_dist,
                'Lakásszám (db)': len(minta),
                'Lefedettség (%)': f"{len(minta)/len(elado.dropna(subset=[col]))*100:.1f}%" if len(elado.dropna(subset=[col])) > 0 else '0%',
                'Medián Ár/m²': minta['nm_ar_huf'].median(),
                'Átlagár (M Ft)': minta['ar_millio_ft'].mean()
            })

df_izokron = pd.DataFrame(izokron_adatok)

fig3 = px.bar(
    df_izokron,
    x='Csomópont',
    y='Medián Ár/m²',
    color='Gyalogos Izokrón',
    barmode='group',
    title='Medián Négyzetméterár a Nemzetközi Gyalogos Izokrón Sávokban (375m / 750m / 1125m)',
    labels={'Medián Ár/m²': 'Medián Fajlagos Ár (Ft/m²)'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450)
fig3.show()

# Összefoglaló táblázat
display(HTML("<div style='max-width: 800px; margin: 15px 0;'>" + df_izokron.round(1).to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

### 4. Interaktív Célpont- és Távolságelemző Pult
Válasszon a légvonalbeli környezeti terhelési távolság vagy a hálózati gyalogos elérhetőségek közül!

In [ ]:
w_cel = widgets.Dropdown(
    options=[
        ('Vasúti pályatest légvonal (zaj/rezgés teher)', 'tavolsag_vasut_m'),
        ('Kőbánya alsó vasútállomás hálózat (séta)', 'tavolsag_vasut_halozati_m'),
        ('Mázsa tér akcióterület hálózat (séta)', 'tavolsag_mazsa_halozati_m'),
        ('Metróállomás hálózat (séta)', 'tavolsag_metro_halozati_m'),
        ('Belváros (Deák tér) hálózat', 'tavolsag_belvaros_halozati_m')
    ],
    value='tavolsag_vasut_m',
    description='Távolság:'
)
w_max_dist = widgets.IntSlider(min=300, max=5000, step=100, value=2000, description='Max táv (m):')

out_dist = widgets.Output()

def frissit_dist(*args):
    col = w_cel.value
    sub = elado[elado[col] <= w_max_dist.value]
    with out_dist:
        clear_output(wait=True)
        fig = px.scatter(
            sub, x=col, y='nm_ar_huf', color='varosresz', trendline='ols',
            title=f'{w_cel.label} vs. Ár/m² (Szűrt minta N={len(sub)})',
            labels={col: f'{w_cel.label} (méter)', 'nm_ar_huf': 'Ár / m² (HUF)'},
            template=PLOTLY_TEMPLATE
        )
        fig.show()

w_cel.observe(frissit_dist, names='value')
w_max_dist.observe(frissit_dist, names='value')

display(widgets.HBox([w_cel, w_max_dist]))
display(out_dist)
frissit_dist()